# Modeling

Imports

In [129]:
import re
import joblib
import optuna
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm import tqdm

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import (
    VarianceThreshold,
    SelectKBest,
    SelectFromModel,
    mutual_info_classif
)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    f1_score,
    accuracy_score
)
from sklearn.model_selection import (
    TimeSeriesSplit,
    train_test_split
)
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from xgboost import XGBClassifier, callback

Load WTI Data

In [130]:
wti = pd.read_parquet(r"C:\Users\02_Florian_Benutzer\Desktop\trump_oil_analysis_v2\00_data\wti_v3.parquet")

Load Tweet Data

In [131]:
tweets = pd.read_parquet(r"C:\Users\02_Florian_Benutzer\Desktop\trump_oil_analysis_v2\00_data\tweets_v6.parquet")

In [132]:
tweets.shape

(15242, 53)

In [133]:
wti.shape

(4206118, 77)

## Train-Validation-Test-Split

In [134]:
tweets_pre2020 = tweets[tweets['timestamp_utc'] < '2020-01-01'] # wird für training und validation verwendet
tweets_post2020 = tweets[tweets['timestamp_utc'] >= '2020-01-01'] # als test set für die out of sample performance

## ML-Pipeline

To Dos:
- feature selection
- feature importance + diagnostik fehlt
- hyperparameter tuning fehlt
- modelle müssen angepasst werden - MLP bewusster konfigurieren etc., early stopping etc. pp
- overfitting detecten und beheben
- andere metriken nehmen
- in dataframe wegschreiben und im nachgang analysierenm, auch feature importance, residen, testing etc.
- Visualisierungen bei timerseries split, learning curves etc.
- Residuenanalye + Diagnostik!
- ggf das finale model nochmal auf kompletten Daten trainieren


# Aktueller Ansatz:

### 1. Config

Test Configs

In [135]:
CONFIG = {
    "outer_splits": 2,
    "inner_splits": 2,
    "gap": 0,

    "targets": ["y_up_2"],  # nur ein Target

    "feature_sets": ["enhanced"],

    "models": ["logreg"],

    "optuna_trials": 1,

    "embedding_pca_components": 30,
}

PROD Configs

In [136]:
# # PCA_COMPONENTS = 20
# # N_SPLITS = 5
# # PURGE_MINUTES = 1440

# TIME_WTI = "timestamp_utc"
# TIME_TWEET = "timestamp_utc"


# CONFIG = {
#     "outer_splits": 5,
#     "inner_splits": 3,
#     "gap": 150,

#     "targets": [c for c in wti.columns if c.startswith("y_")],
#     "feature_sets": ["baseline", "enhanced"],
#     "models": ["logreg", "xgboost", "mlp"],

#     "optuna_trials": 30,

#     # "top_k_features": 0.7,

#     # PCA for embeddings
#     "embedding_pca_components": 30,

#     # Mutual Information
#    #  "mi_top_k": 150,    
# }

### 2. Embedding Parser

In [137]:
def parse_embedding(x):
    return np.fromstring(x.strip("[]"), sep=" ")

### 3. Tweet Preprocessing + Embeddings

In [138]:
def prepare_tweets(df):

    df = df.copy()

    df[TIME_TWEET] = pd.to_datetime(df[TIME_TWEET], utc=True)

    df["embedding_vec"] = df["embedding"].apply(parse_embedding)

    emb = np.vstack(df["embedding_vec"].values)

    emb_cols = [f"emb_{i}" for i in range(emb.shape[1])]
    emb_df = pd.DataFrame(emb, columns=emb_cols)

    df = pd.concat([df.reset_index(drop=True), emb_df], axis=1)

    return df

### 4. Tweet Aggregation

Da es zum Teil mehrere Tweets innerhalb einer Minute gibt

In [139]:
def aggregate_tweets(df):

    df = df.copy()

    df["minute"] = df[TIME_TWEET].dt.floor("min")

    emb_cols = [c for c in df.columns if c.startswith("emb_")]

    agg_dict = {

        # =========================
        # FinBERT
        # =========================
        "finbert_positive": "mean",
        "finbert_neutral": "mean",
        "finbert_negative": "mean",
        "finbert_sentiment_score": "mean",
        "finbert_entropy": "mean",
        "finbert_confidence": "max",
        "finbert_polarization": "max",

        # =========================
        # Financial NLP Scores
        # =========================
        "financial_uncertainty_score": "max",
        "financial_risk_sentiment": "mean",

        # =========================
        # Text Style
        # =========================
        "caps_ratio": "mean",
        "exclamation_count": "max",
        "exclamation_count_log": "max",

        "tweet_length": "mean",
        "tweet_length_log": "mean",

        "market_aggression_index": "max",

        # =========================
        # Tweet Dynamics
        # =========================
        "time_since_last_tweet_min": "min",

        "rolling_tweet_frequency_60m": "max",
        "rolling_tweet_frequency_6h": "max",
        "rolling_tweet_frequency_24h": "max",

        "tweet_burst_indicator": "max",
        "tweet_acceleration_6h": "max",

        # =========================
        # Sentiment Dynamics
        # =========================
        "sentiment_delta_vs_previous": "mean",
        "rolling_sentiment_mean_60m": "mean",
        "rolling_sentiment_std_60m": "mean",

        # =========================
        # Presidency Regime
        # =========================
        "is_trump_president": "max",

        # =========================
        # Oil / Macro Scores
        # =========================
        "wti_bullish_score": "max",
        "wti_bearish_score": "max",

        "energy_supply_score": "max",
        "geopolitical_oil_risk_score": "max",
        "usd_strength_pressure_score": "max",
        "fed_monetary_policy_score": "max",
        "risk_sentiment_score": "max",

        "trump_energy_policy_score": "max",
        "trump_market_shock_language_score": "max",
        "policy_shock_score": "max",

        # =========================
        # Topic Metrics
        # =========================
        "topic_concentration": "mean",

        "topic_energy_supply": "max",
        "topic_fed_monetary_policy": "max",
        "topic_geopolitical_oil_risk": "max",
        "topic_policy_shock": "max",
        "topic_risk_sentiment": "max",
        "topic_trump_energy_policy": "max",
        "topic_trump_market_shock_language": "max",
        "topic_usd_strength_pressure": "max",
        "topic_wti_bearish": "max",
        "topic_wti_bullish": "max",

        # =========================
        # Semantic Change
        # =========================
        "semantic_global_novelty": "max",
        "semantic_local_change": "max",
        "semantic_rolling_novelty": "max",
    }

    # =========================
    # Embeddings
    # =========================
    for c in emb_cols:
        agg_dict[c] = "mean"

    agg = (
        df.groupby("minute")
          .agg(agg_dict)
          .reset_index()
    )

    return agg

### 5. Leakage-Free WTI Mapping

Wird ein Tweet um 12:37:21 gepostet, mappt die Funktion den WTI Kurs von 12:36:00 hinzu. Das soll Data Leakage verhindern, da wir den WTI Daten nicht entnehmen können, ob der Kurs um 12:37:00 den Kurs von 12:36:00 - 12:36:59 oder 12:37:00 bis 12:37:59 meint

In [140]:
def map_to_wti(tweet_agg, wti):

    tweet_agg = tweet_agg.copy()
    wti = wti.copy()

    tweet_agg[TIME_TWEET] = pd.to_datetime(tweet_agg["minute"], utc=True)
    wti[TIME_WTI] = pd.to_datetime(wti[TIME_WTI], utc=True)

    # KEY RULE:
    # tweet -> previous completed minute candle
    tweet_agg["wti_time"] = tweet_agg["minute"] - pd.Timedelta(minutes=1)

    df = tweet_agg.merge(
        wti,
        left_on="wti_time",
        right_on=TIME_WTI,
        how="left"
    )

    df = df.dropna(subset=["open"])

    return df

### 6. Feature / Target Split

In [141]:
TARGETS = [c for c in wti.columns if c.startswith("y_")]

def split_xy(df):

    y = df[TARGETS]
    X = df.drop(columns=TARGETS)

    return X, y

### 7. Evaluation

In [142]:
def evaluate(y_true, prob):

    return {
        "roc_auc": roc_auc_score(y_true, prob),
        # "f1": f1_score(y_true, pred),
        # "acc": accuracy_score(y_true, pred)
    }

### 8. Clean Features

In [143]:
def clean_features(df):

    # print(df.isna().sum()[df.isna().sum() > 0])

    # nur numerische Features
    df = df.select_dtypes(include=[np.number])

    # df = df.dropna()

    return df


### 9. Build Dataset

Datensatz wird erstellt, der später in den Training Loop geht

In [144]:
def build_dataset(wti_df, tweets_df):

    tweets_df = prepare_tweets(tweets_df)
    tweets_agg = aggregate_tweets(tweets_df)

    df = map_to_wti(tweets_agg, wti_df)
    df = df.sort_values("minute").reset_index(drop=True)

    X, y = split_xy(df)

    return df, X, y

## 10. Feature Sets

Feature-Sets für den Baseline und den um Tweet Infos angereicherten Datensatz

In [145]:
def get_feature_sets(X):

    baseline_cols = [
        'open', 'high', 'low', 'close', 'was_missing',
        'log_close', 'ret_1', 'hl_range', 'oc_return', 'ret_5', 'ret_15',
        'ret_30', 'ret_60', 'ret_240', 'mom_5', 'mom_15', 'mom_60',
        'vol_close_5', 'vol_parkinson_5', 'vol_close_15', 'vol_parkinson_15',
        'vol_close_30', 'vol_parkinson_30', 'vol_close_60', 'vol_parkinson_60',
        'vol_close_240', 'vol_parkinson_240', 'vol_gk', 'vol_ratio_5_60',
        'sma_5', 'ema_5', 'dist_ema_5', 'sma_15', 'ema_15', 'dist_ema_15',
        'sma_60', 'ema_60', 'dist_ema_60', 'sma_240', 'ema_240', 'dist_ema_240',
        'hl_momentum', 'body', 'upper_wick', 'lower_wick', 'abs_ret',
        'vol_cluster', 'vol_5', 'jump', 'hour', 'dayofweek', 'hour_sin',
        'hour_cos', 'dow_sin', 'dow_cos', 'us_session', 'vol_rank',
        'vol_regime', 'trend_60', 'trend_regime'
    ]

    return {
        "baseline": clean_features(X[baseline_cols].copy()),
        "enhanced": clean_features(X.copy())
    }

### 11. Time Series Split

Zeitliche Abhängigkeiten bleiben erhalten und es wird nicht geshuffled

In [146]:
def outer_cv():
    return TimeSeriesSplit(
        n_splits=CONFIG["outer_splits"],
        gap=CONFIG["gap"]
    )

def inner_cv():
    return TimeSeriesSplit(
        n_splits=CONFIG["inner_splits"],
        gap=CONFIG["gap"]
    )

### 12. Initialisierung der ML-Models

In [147]:
def build_model(name, params):

    if name == "logreg":
        return LogisticRegression(
            penalty="elasticnet",
            solver="saga",
            max_iter=5000,
            class_weight="balanced",
            **params
        )

    if name == "xgboost":
        return XGBClassifier(
            eval_metric="logloss",
            tree_method="hist",
            random_state=42,
            n_estimators = 5000,
            early_stopping_rounds = 50,
            #verbose = False,
            **params
        )

    if name == "mlp":
        return MLPClassifier(
            max_iter=2000,
            early_stopping=True,
            validation_fraction=0.2,
            n_iter_no_change=20,
            random_state=42,
            **params
        )

# Ab Hier muss Pipeline überarbeitet werden

### 13. Correlation

entfernt stark korrelierende Features

In [148]:
class CorrelationFilter(BaseEstimator, TransformerMixin):

    def __init__(self, threshold=0.95):
        self.threshold = threshold

    def fit(self, X, y=None):

        X_df = (
            pd.DataFrame(X)
            if not isinstance(X, pd.DataFrame)
            else X
        )

        corr = X_df.corr().abs()

        upper = corr.where(
            np.triu(
                np.ones(corr.shape),
                k=1
            ).astype(bool)
        )

        self.to_drop_ = [
            col for col in upper.columns
            if any(upper[col] > self.threshold)
        ]

        self.feature_names_in_ = np.array(X_df.columns)

        self.keep_mask_ = ~np.isin(
            self.feature_names_in_,
            self.to_drop_
        )

        return self


    def transform(self, X):

        X_df = pd.DataFrame(
            X,
            columns=self.feature_names_in_
        )

        return X_df.loc[:, self.keep_mask_].values

In [149]:
# class CorrelationFilter(BaseEstimator, TransformerMixin):

#     def __init__(self, threshold=0.95):
#         self.threshold = threshold

#     def fit(self, X, y=None):

#         # ensure DataFrame for correlation step
#         X_df = pd.DataFrame(X) if not isinstance(X, pd.DataFrame) else X

#         corr = X_df.corr().abs()

#         upper = corr.where(
#             np.triu(np.ones(corr.shape), k=1).astype(bool)
#         )

#         self.to_drop_ = [
#             col for col in upper.columns
#             if any(upper[col] > self.threshold)
#         ]

#         self.feature_names_in_ = list(X_df.columns)

#         return self

#     def transform(self, X):

#         X_df = pd.DataFrame(X, columns=getattr(self, "feature_names_in_", None))

#         X_df = X_df.drop(columns=self.to_drop_, errors="ignore")

#         return X_df

Embedding Columns

In [150]:
def get_embedding_cols(X):

    return [
        c
        for c in X.columns
        if c.startswith("emb_")
    ]


### 14. Preprocessor

Definiert Pipeline Schritte, die Embedding-Features werden PCA komprimiert und skaliert, die restlichen Features nur skaliert.

In [151]:
# feature_set, embedding_cols, feature_names

def build_preprocessor(emb_cols, non_emb_cols):

    # numeric_cols = [
    #     c for c in feature_names
    #     if c not in embedding_cols
    # ]

    embeddings_pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=CONFIG["embedding_pca_components"]))
    ])

    # numeric_pipe = "passthrough"
    numeric_pipe = Pipeline([
        ("scaler", StandardScaler())
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("embeddings", embeddings_pipe, emb_cols),
            ("numeric", numeric_pipe, non_emb_cols),
        ],
        remainder="drop",
        verbose_feature_names_out=False
    )

    # optional baseline-only feature filtering
    # if feature_set == "baseline":
    #     return Pipeline([
    #         ("preprocessor", preprocessor),
    #         ("variance", VarianceThreshold(0.01)),
    #         ("corr", CorrelationFilter())
    #     ])

    # production path: still include correlation filtering (recommended)
    return Pipeline([
        ("preprocessor", preprocessor),
        ("variance", VarianceThreshold(0.01)),
        ("corr", CorrelationFilter())
    ])

### 15. Pipeline

Führt für jedes Model die Preprocessing Schritte durch + anschließender Model fit

In [152]:
def make_pipeline(
    model_name,
    params,
    emb_cols,
    non_emb_cols
    # feature_set,
    # embedding_cols,
    # feature_names
):

    model = build_model(model_name, params)

    steps = []

    # -------------------------
    # PREPROCESSOR (UNIFIED)
    # -------------------------
    steps.append((
        "preprocessor",
        build_preprocessor(
            emb_cols,
            non_emb_cols
            # feature_set,
            # embedding_cols,
            # feature_names
        )
    ))

    # -------------------------
    # MODEL-SPECIFIC LOGIC
    # -------------------------
    # if model_name == "logreg":

    #     steps.append(("selector", logreg_selector()))
    #     #steps.append(("scaler", StandardScaler()))

    # elif model_name == "xgboost":

    #     steps.append(("selector", xgb_selector()))
    #     # no scaling needed

    # elif model_name == "mlp":

    #     steps.append(("selector", mlp_selector()))
    #     #steps.append(("scaler", StandardScaler()))

    # -------------------------
    # MODEL
    # -------------------------
    steps.append(("model", model))

    return Pipeline(steps)

In [153]:
# def make_pipeline(model_name, params):

#     model = build_model(model_name, params)

#     steps = []

#     # scaling only for linear / neural
#     if model_name in ["logreg", "mlp"]:
#         steps.append(("scaler", StandardScaler()))

#     # feature selection only for tree models
#     if model_name == "xgboost":
#         steps.append(("selector", TreeFeatureSelector(model)))

#     steps.append(("model", model))

#     return Pipeline(steps)

### 16. Optuna - Model Tuning

- automatische Hyperparameter-Optimierung mittels Optuna
- Features werden nicht preselcted und nur über die Regularisierungs-Parameter "entfernt" (alos Gewicht = 0)

Optuna (inner CV only)

In [154]:
def tune_model(model_name, X_train, y_train, emb_cols, non_emb_cols, inner):

    def objective(trial):

        if model_name == "logreg":
            params = {
                "C": trial.suggest_float("C", 1e-4, 1e2, log=True),  # enger & stärker regularisiert # vorher war 1.0 upper bound
                "l1_ratio": trial.suggest_float("l1_ratio", 0.0, 1.0)  # wenn solver elasticnet
            }

        elif model_name == "xgboost":
            params = {
                # "n_estimators": trial.suggest_int("n_estimators", 200, 800),
            
                "max_depth": trial.suggest_int("max_depth", 2, 6),   # <<< KEY CHANGE
            
                "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.05, log=True),
            
                "subsample": trial.suggest_float("subsample", 0.6, 0.9),
            
                "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 0.9),

                # 🔥 STRONG regularization (currently missing)
                "min_child_weight": trial.suggest_float("min_child_weight", 1, 10),
            
                "gamma": trial.suggest_float("gamma", 0, 5),
            
                "reg_lambda": trial.suggest_float("reg_lambda", 1, 20),
            
                "reg_alpha": trial.suggest_float("reg_alpha", 0, 5),            
            }

        elif model_name == "mlp":
            h1 = trial.suggest_int("h1", 16, 64)
            h2 = trial.suggest_int("h2", 8, 32)

            params = {
                "hidden_layer_sizes": (h1, h2),

                "alpha": trial.suggest_float("alpha", 1e-4, 1e-1, log=True),

                "learning_rate_init": trial.suggest_float("learning_rate_init", 1e-4, 5e-3, log=True),

                "batch_size": trial.suggest_categorical("batch_size",[32, 64, 128, 256]),

                "activation": trial.suggest_categorical("activation", ["relu", "tanh"])
            }

        scores = []
        # inner = inner_cv()

        for tr, va in inner.split(X_train):

            X_tr, X_va = X_train.iloc[tr], X_train.iloc[va]
            y_tr, y_va = y_train.iloc[tr], y_train.iloc[va]
            
            if model_name == "xgboost":
                # -------------------------
                # Train -> Train_ES Split
                # -------------------------
                es_size = int(len(X_tr) * 0.2)

                X_fit = X_tr.iloc[:-es_size]
                y_fit = y_tr.iloc[:-es_size]

                X_es = X_tr.iloc[-es_size:]
                y_es = y_tr.iloc[-es_size:]

                # -------------------------
                # Preprocessing
                # -------------------------
                preprocessor = build_preprocessor(emb_cols, non_emb_cols)

                X_fit_prep = preprocessor.fit_transform(X_fit, y_fit)

                X_es_prep = preprocessor.transform(X_es)

                X_va_prep = preprocessor.transform(X_va)

                # -------------------------
                # Model
                # -------------------------
                model = build_model(model_name, params)

                model.fit(
                    X_fit_prep,
                    y_fit,
                    eval_set=[(X_es_prep, y_es)],
                    verbose=False
                )

                prob = model.predict_proba(X_va_prep)[:, 1]
                                
            else:
                pipe = make_pipeline(model_name, params, emb_cols, non_emb_cols)
                pipe.fit(X_tr, y_tr)

                prob = pipe.predict_proba(X_va)[:, 1]
            scores.append(roc_auc_score(y_va, prob))

        return np.mean(scores)

    study = optuna.create_study(direction="maximize", pruner=optuna.pruners.MedianPruner())
    study.optimize(objective, n_trials=CONFIG["optuna_trials"])

    best_params = study.best_params

    if model_name == "mlp":
        best_params = best_params.copy()
        best_params["hidden_layer_sizes"] = (
            best_params.pop("h1"),
            best_params.pop("h2")
        )


    return best_params

### 17. Finaler Model Fit

Hier wird das jeweilige ML-Model mit den optimierten Hyperparmaetern auf den CV-Fold gefittet und validiert

Outer Fold Training (Unbiased)

In [155]:
def run_fold(model_name, best_params, X_train, X_val, y_train, y_val, emb_cols, non_emb_cols):

    if model_name == "xgboost":
        # -------------------------
        # Train -> Train_ES Split
        # -------------------------
        es_size = int(len(X_train) * 0.2)

        X_fit = X_train.iloc[:-es_size]
        y_fit = y_train.iloc[:-es_size]

        X_es = X_train.iloc[-es_size:]
        y_es = y_train.iloc[-es_size:]

        # -------------------------
        # Preprocessing
        # -------------------------
        preprocessor = build_preprocessor(emb_cols, non_emb_cols)

        X_fit_prep = preprocessor.fit_transform(X_fit, y_fit)

        X_es_prep = preprocessor.transform(X_es)

        X_va_prep = preprocessor.transform(X_val)

        X_train_prep = preprocessor.transform(X_train)

        # -------------------------
        # Model
        # -------------------------
        pipe = build_model(model_name, best_params)

        pipe.fit(
            X_fit_prep,
            y_fit,
            eval_set=[(X_es_prep, y_es)],
            verbose=False
        )

        prob_train = pipe.predict_proba(X_train_prep)[:, 1]
        prob_val = pipe.predict_proba(X_va_prep)[:, 1]

        return pipe, preprocessor, prob_train, prob_val

    else:

        pipe = make_pipeline(model_name, best_params, emb_cols, non_emb_cols)

        pipe.fit(X_train, y_train)

        prob_train = pipe.predict_proba(X_train)[:, 1]
        prob_val = pipe.predict_proba(X_val)[:, 1]

        # pred_train = (prob_train > 0.5).astype(int)
        # pred_val = (prob_val > 0.5).astype(int)

        return pipe, None, prob_train, prob_val

### 18. Experiment Loop (Full nested CV)

In dieser Training Loop werden die Schritte 1-17 automatisiert für die Kombinationen aus Target, Feature-Set und ML-Model durchgeführt und die Ergebnisse + Modellkonfigurationen weggeschrieben

- brauche noch mechanismus um die ausgewählten features mit wegzuschreiben
- timestamp auch wegschreiben der prediction

Feature Importance:
- ab pipe.fit() in run_fold werden die eingangsdaten des df transformiert

In [156]:
def run_experiment(wti_df, tweets_df):

    df, X, y = build_dataset(wti_df, tweets_df)
    feature_sets = get_feature_sets(X)

    outer = outer_cv()
    inner = inner_cv()

    results = []
    predictions = []
    models = {}

    for target in CONFIG["targets"]:

        y_t = y[target]

        for fs_name, X_fs in feature_sets.items():
            emb_cols = [c for c in X_fs.columns if c.startswith("emb_")]
            non_emb_cols = [c for c in X_fs.columns if c not in emb_cols]
            
            for model_name in CONFIG["models"]:

                # -------------------------
                # OUTER CV (EVALUATION)
                # -------------------------
                for fold, (tr, va) in enumerate(outer.split(X_fs)):

                    X_train, X_val = X_fs.iloc[tr], X_fs.iloc[va]
                    y_train, y_val = y_t.iloc[tr], y_t.iloc[va]

                    # -------------------------
                    # INNER CV (TUNING)
                    # -------------------------
                    best_params = tune_model(
                        model_name,
                        X_train,
                        y_train,
                        emb_cols,
                        non_emb_cols,
                        inner
                    )

                    model, preprocessor, prob_train, prob_val = run_fold(
                        model_name,
                        best_params,
                        X_train,
                        X_val,
                        y_train,
                        y_val,
                        emb_cols,
                        non_emb_cols
                    )

                    #scores_train = evaluate(y_train, prob_train)
                    #scores_val = evaluate(y_val, prob_val)

                    scores_train = roc_auc_score(y_train, prob_train)
                    scores_val = roc_auc_score(y_val, prob_val)

                    results.append({
                        "target": target,
                        "feature_set": fs_name,
                        "model": model_name,
                        "fold": fold,
                        "train_auc": scores_train,
                        "val_auc": scores_val,
                        "y_train_true": y_train.values,
                        "y_train_prob": prob_train,
                        "y_train_index": y_train.index,
                        "y_val_true": y_val.values,
                        "y_val_prob": prob_val,
                        "y_val_index": y_val.index,
                        "params": best_params
                    })

                    # predictions.append({
                    #     "target": target,
                    #     "feature_set": fs_name,
                    #     "model": model_name,
                    #     "fold": fold,
                    #     "y_true": y_val.values,
                    #     "y_pred": pred,
                    #     "y_prob": prob,
                    #     "index": y_val.index
                    # })

                    models[(target, fs_name, model_name, fold)] = {"model": model, "preprocessor": preprocessor}

    return results, models

## 19. Function Call: Training Loop

In [157]:
results, models = run_experiment(wti, tweets_pre2020)

C:\Users\02_Florian_Benutzer\AppData\Local\Temp\ipykernel_376\2612417994.py:113: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  .reset_index()
[I 2026-06-16 19:33:38,978] A new study created in memory with name: no-name-78b6cb75-0b2a-44db-8321-d5359df08ece
c:\Users\02_Florian_Benutzer\Desktop\trump_oil_analysis_v2\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\02_Florian_Benutzer\Desktop\trump_oil_analysis_v2\.venv\Lib\site-

In [158]:
results

[{'target': 'y_up_2',
  'feature_set': 'baseline',
  'model': 'logreg',
  'fold': 0,
  'train_auc': 0.7489591654588649,
  'val_auc': 0.7592192972059204,
  'y_train_true': array([0, 0, 0, ..., 0, 0, 0], shape=(3031,)),
  'y_train_prob': array([0.43961466, 0.46819499, 0.6343287 , ..., 0.18884766, 0.18481844,
         0.18335151], shape=(3031,)),
  'y_train_index': RangeIndex(start=0, stop=3031, step=1),
  'y_val_true': array([0, 0, 0, ..., 0, 0, 0], shape=(3030,)),
  'y_val_prob': array([0.17977595, 0.15753478, 0.15320368, ..., 0.5068203 , 0.73044648,
         0.64865066], shape=(3030,)),
  'y_val_index': RangeIndex(start=3031, stop=6061, step=1),
  'params': {'C': 0.004749289877112669, 'l1_ratio': 0.30802242203788954}},
 {'target': 'y_up_2',
  'feature_set': 'baseline',
  'model': 'logreg',
  'fold': 1,
  'train_auc': 0.6914082469856233,
  'val_auc': 0.7115297822024683,
  'y_train_true': array([0, 0, 0, ..., 0, 0, 0], shape=(6061,)),
  'y_train_prob': array([0.49540057, 0.50167019, 0.50

# Extrahierung der Coefficients für LogReg!

In [208]:
model = models[("y_up_2", "enhanced", "logreg", 1)]["model"] # .named_steps["preprocessor"].get_feature_names_out()

In [209]:
# Namen nach ColumnTransformer
names = (
    model
    .named_steps["preprocessor"]
    .named_steps["preprocessor"]
    .get_feature_names_out()
)

In [210]:
names

array(['pca0', 'pca1', 'pca2', 'pca3', 'pca4', 'pca5', 'pca6', 'pca7',
       'pca8', 'pca9', 'pca10', 'pca11', 'pca12', 'pca13', 'pca14',
       'pca15', 'pca16', 'pca17', 'pca18', 'pca19', 'pca20', 'pca21',
       'pca22', 'pca23', 'pca24', 'pca25', 'pca26', 'pca27', 'pca28',
       'pca29', 'finbert_positive', 'finbert_neutral', 'finbert_negative',
       'finbert_sentiment_score', 'finbert_entropy', 'finbert_confidence',
       'finbert_polarization', 'financial_uncertainty_score',
       'financial_risk_sentiment', 'caps_ratio', 'exclamation_count',
       'exclamation_count_log', 'tweet_length', 'tweet_length_log',
       'market_aggression_index', 'time_since_last_tweet_min',
       'rolling_tweet_frequency_60m', 'rolling_tweet_frequency_6h',
       'rolling_tweet_frequency_24h', 'tweet_burst_indicator',
       'tweet_acceleration_6h', 'sentiment_delta_vs_previous',
       'rolling_sentiment_mean_60m', 'rolling_sentiment_std_60m',
       'is_trump_president', 'wti_bullish_score'

In [211]:
variance_mask = (
    model
    .named_steps["preprocessor"]
    .named_steps["variance"]
    .get_support()
)

after_variance = names[variance_mask]

In [212]:
corr_mask = (
    model
    .named_steps["preprocessor"]
    .named_steps["corr"]
    .keep_mask_
)

final_features = after_variance[corr_mask]

In [213]:
len(final_features)

114

In [214]:
model_logreg = model.named_steps["model"]

coefs = model_logreg.coef_[0]

In [215]:
model_logreg.coef_[0]

array([ 0.00472367, -0.00476183,  0.00286552,  0.00037928,  0.        ,
        0.        ,  0.00974395,  0.        , -0.01643888,  0.        ,
        0.        ,  0.        ,  0.        , -0.02231857, -0.00087289,
        0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        ,  0.  

In [216]:
len(coefs)

114

In [217]:
importance = pd.DataFrame({
    "feature": final_features,
    "coefficient": coefs,
    "importance_abs": np.abs(coefs)
})

importance = (
    importance
    .sort_values(
        "importance_abs",
        ascending=False
    )
)

In [221]:
print(importance.to_string())

                               feature  coefficient  importance_abs
75                         was_missing    -0.369078        0.369078
105                          dayofweek    -0.203006        0.203006
94                      vol_ratio_5_60     0.187429        0.187429
88                        vol_close_30     0.131700        0.131700
91                       vol_close_240     0.128282        0.128282
112                         vol_regime     0.101152        0.101152
111                           vol_rank     0.053526        0.053526
108                            dow_sin     0.043394        0.043394
107                           hour_cos    -0.039493        0.039493
113                       trend_regime     0.037526        0.037526
13                               pca13    -0.022319        0.022319
8                                 pca8    -0.016439        0.016439
6                                 pca6     0.009744        0.009744
1                                 pca1    -0.004

In [ ]:
raise SystemExit("Notebook hier stoppen")

SystemExit: Notebook hier stoppen

c:\Users\02_Florian_Benutzer\Desktop\trump_oil_analysis_v2\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3756: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [194]:
 results_df = pd.DataFrame(results)

In [195]:
results_df

,target,feature_set,model,fold,train_auc,val_auc,y_train_true,y_train_prob,y_train_index,y_val_true,y_val_prob,y_val_index,params
0,y_up_2,baseline,logreg,0,0.748959,0.759219,"[0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, ...","[0.43961466161538737, 0.4681949945466102, 0.63...","RangeIndex(start=0, stop=3031, step=1)","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.1797759539032096, 0.15753477661298051, 0.15...","RangeIndex(start=3031, stop=6061, step=1)","{'C': 0.004749289877112669, 'l1_ratio': 0.3080..."
1,y_up_2,baseline,logreg,1,0.691408,0.711530,"[0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, ...","[0.49540057202186005, 0.5016701933676905, 0.50...","RangeIndex(start=0, stop=6061, step=1)","[0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.5016701933676905, 0.5016701933676905, 0.501...","RangeIndex(start=6061, stop=9091, step=1)","{'C': 0.00025280958250317547, 'l1_ratio': 0.28..."
2,y_up_2,enhanced,logreg,0,0.500000,0.500000,"[0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, ...","[0.5091401906959252, 0.5091401906959252, 0.509...","RangeIndex(start=0, stop=3031, step=1)","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.5091401906959252, 0.5091401906959252, 0.509...","RangeIndex(start=3031, stop=6061, step=1)","{'C': 0.0002762225147510869, 'l1_ratio': 0.838..."
3,y_up_2,enhanced,logreg,1,0.755709,0.753640,"[0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, ...","[0.3469665352372852, 0.48757970098164094, 0.62...","RangeIndex(start=0, stop=6061, step=1)","[0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.6127141366766204, 0.5515872434341901, 0.654...","RangeIndex(start=6061, stop=9091, step=1)","{'C': 0.0030201278953447663, 'l1_ratio': 0.474..."


In [ ]:
# results_df = pd.DataFrame(results)

# results_df.to_pickle(r"C:\Users\02_Florian_Benutzer\Desktop\trump_oil_analysis_v2\results\results.pkl")

In [ ]:
results_df.head(30)

,target,feature_set,model,fold,train_auc,val_auc,y_train_true,y_train_prob,y_train_index,y_val_true,y_val_prob,y_val_index,params
0,y_up_2,baseline,logreg,0,0.769507,0.729295,"[0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, ...","[0.556625418631707, 0.5533669644248155, 0.7903...","RangeIndex(start=0, stop=1366, step=1)","[0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, ...","[0.07629702771168176, 0.07347127999238501, 0.0...","RangeIndex(start=1516, stop=3031, step=1)","{'C': 0.03325365192708734, 'l1_ratio': 0.24725..."
1,y_up_2,baseline,logreg,1,0.756337,0.758669,"[0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, ...","[0.47466228276081035, 0.4475103266893604, 0.67...","RangeIndex(start=0, stop=2881, step=1)","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.19053686800309289, 0.17910399928108262, 0.1...","RangeIndex(start=3031, stop=4546, step=1)","{'C': 0.0005320119641144345, 'l1_ratio': 0.002..."
2,y_up_2,baseline,logreg,2,0.754636,0.760826,"[0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, ...","[0.4685345757183949, 0.4471983057187812, 0.681...","RangeIndex(start=0, stop=4396, step=1)","[0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, ...","[0.6059883799146041, 0.6388714431363421, 0.641...","RangeIndex(start=4546, stop=6061, step=1)","{'C': 0.00046967397818246356, 'l1_ratio': 0.00..."
3,y_up_2,baseline,logreg,3,0.756599,0.747749,"[0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, ...","[0.4241735011292147, 0.4356246227468943, 0.595...","RangeIndex(start=0, stop=5911, step=1)","[0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.5757406721157711, 0.515553320468328, 0.6314...","RangeIndex(start=6061, stop=7576, step=1)","{'C': 0.0001690849589743452, 'l1_ratio': 0.004..."
4,y_up_2,baseline,logreg,4,0.756860,0.762365,"[0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, ...","[0.3968043623350643, 0.4224787378538347, 0.636...","RangeIndex(start=0, stop=7426, step=1)","[0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, ...","[0.6791291494723022, 0.6075739733238025, 0.701...","RangeIndex(start=7576, stop=9091, step=1)","{'C': 0.0005562142282933535, 'l1_ratio': 0.011..."
5,y_up_2,baseline,xgboost,0,0.826213,0.738138,"[0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, ...","[0.18761, 0.21650578, 0.4280719, 0.33188894, 0...","RangeIndex(start=0, stop=1366, step=1)","[0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, ...","[0.030198442, 0.030198442, 0.030198442, 0.0304...","RangeIndex(start=1516, stop=3031, step=1)","{'max_depth': 5, 'learning_rate': 0.0116187969..."
6,y_up_2,baseline,xgboost,1,0.805208,0.749961,"[0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, ...","[0.22712098, 0.22308742, 0.40595317, 0.3226557...","RangeIndex(start=0, stop=2881, step=1)","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.02125852, 0.019635865, 0.01562297, 0.015622...","RangeIndex(start=3031, stop=4546, step=1)","{'max_depth': 3, 'learning_rate': 0.0218716624..."
7,y_up_2,baseline,xgboost,2,0.779976,0.770655,"[0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, ...","[0.22834806, 0.22525907, 0.39227098, 0.3517349...","RangeIndex(start=0, stop=4396, step=1)","[0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, ...","[0.38722715, 0.37460423, 0.36941743, 0.4102834...","RangeIndex(start=4546, stop=6061, step=1)","{'max_depth': 2, 'learning_rate': 0.0459872387..."
8,y_up_2,baseline,xgboost,3,0.784994,0.754836,"[0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, ...","[0.22686227, 0.19268991, 0.4250083, 0.35449395...","RangeIndex(start=0, stop=5911, step=1)","[0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.36851943, 0.3399974, 0.44917807, 0.34617, 0...","RangeIndex(start=6061, stop=7576, step=1)","{'max_depth': 2, 'learning_rate': 0.0073232453..."
9,y_up_2,baseline,xgboost,4,0.781511,0.774867,"[0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, ...","[0.20672733, 0.17168632, 0.38557833, 0.3333807...","RangeIndex(start=0, stop=7426, step=1)","[0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, ...","[0.41982177, 0.3817415, 0.4065129, 0.4670805, ...","RangeIndex(start=7576, stop=9091, step=1)","{'max_depth': 2,

In [ ]:
import pandas as pd

test11 = pd.read_pickle(r"C:\Users\02_Florian_Benutzer\Desktop\trump_oil_analysis_v2\results\results.pkl")


In [ ]:
test11.shape

(480, 13)

In [ ]:
models

NameError: name 'models' is not defined

In [ ]:
import joblib

joblib.dump(models, r"C:\Users\02_Florian_Benutzer\Desktop\trump_oil_analysis_v2\results\models.joblib")

models_loaded = joblib.load(r"C:\Users\02_Florian_Benutzer\Desktop\trump_oil_analysis_v2\results\models.joblib")

#key = list(models.keys())[0]

# pred1 = models[key]["model"].predict_proba(X_test)
# pred2 = models_loaded[key]["model"].predict_proba(X_test)

# assert (pred1 == pred2).all()

In [ ]:
print("test")

In [ ]:
models.keys()

In [ ]:
models_loaded[("y_up_2","enhanced","xgboost", 1)]["model"].feature_importances_

array([0.00950808, 0.01078813, 0.01112744, 0.01005743, 0.00967642,
       0.01225126, 0.01031962, 0.        , 0.00971443, 0.01071288,
       0.00965539, 0.        , 0.00800249, 0.01027906, 0.00886815,
       0.00804731, 0.00997279, 0.00800014, 0.00846142, 0.01139115,
       0.01158872, 0.00954795, 0.        , 0.0098182 , 0.01122806,
       0.0085939 , 0.0085624 , 0.00957525, 0.0093171 , 0.01078434,
       0.00986838, 0.        , 0.        , 0.        , 0.00872902,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.00997042, 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.0097029 , 0.00838354, 0.        ,
       0.        , 0.        , 0.00943878, 0.00996246, 0.00993864,
       0.0100954 , 0.        , 0.01201643, 0.        , 0.        ,
       0.00885082, 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.00932988, 0.01068209, 0.00966332, 0.01046

### Post Analysis Module

Model Ranking

In [ ]:
def rank_models(results_df):
    return results_df.groupby(["model", "feature_set"]).mean()

Stability Analysis

In [ ]:
def fold_stability(results_df):
    return results_df.groupby("model")["roc_auc"].std()

Prediction Drift

In [ ]:
def plot_time_performance(predictions_df):

    df = pd.DataFrame(predictions_df)
    df["correct"] = df["y_pred"] == df["y_true"]

    return df.groupby("fold")["correct"].mean()

SHAP (after training only)

In [ ]:
import shap

def shap_analysis(model, X):

    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X)

    shap.summary_plot(shap_values, X)